<a href="https://colab.research.google.com/github/Praharshita1275/star_pnt/blob/main/Parking_Slot_Detection_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚗 Parking Slot Detection and Tracking - STAR PNT Challenge 2
This Google Colab notebook guides you through training a YOLOv8 model to detect and classify parking slots.

In [ ]:
!pip install ultralytics opencv-python matplotlib pandas -q

In [ ]:
from ultralytics import YOLO
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

## 📁 Upload Dataset (.zip format from Roboflow or your own YOLO-format dataset)

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os
with zipfile.ZipFile(next(iter(uploaded)), 'r') as zip_ref:
    zip_ref.extractall("parking_dataset")

In [ ]:
!ls parking_dataset

## 🧠 Train YOLOv8 Model

In [ ]:
model = YOLO('yolov8s.pt')
results = model.train(
    data='parking_dataset/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    name='parking-model'
)

## 📷 Upload a Test Image for Inference

In [ ]:
uploaded_test = files.upload()
test_image_path = next(iter(uploaded_test))

## 🔍 Run Inference

In [ ]:
predict_results = model.predict(source=test_image_path, save=True, conf=0.5)

## 🧮 Count Slots and Save Results to CSV

In [ ]:
def count_parking_slots(results):
    total_slots = len(results[0].boxes)
    occupied_slots = 0
    for box in results[0].boxes:
        class_id = int(box.cls[0].item())
        if class_id == 0:
            occupied_slots += 1
    available_slots = total_slots - occupied_slots
    return total_slots, occupied_slots, available_slots

total, occupied, available = count_parking_slots(predict_results)

df = pd.DataFrame([{
    'Total Slots': total,
    'Occupied Slots': occupied,
    'Available Slots': available
}])
df.to_csv('parking_output.csv', index=False)
print("✅ CSV Saved as parking_output.csv")

In [ ]:
from google.colab import files
files.download("parking_output.csv")